In [3]:
# 0. Attach a lakehouse before running this notebook

import os

# 1. Define the local Fabric mount directory path for your Lakehouse files
directory_path = "/lakehouse/default/Files/"
file_path = os.path.join(directory_path, "Transactions_raw.csv")

# Ensure the target directory exists
os.makedirs(directory_path, exist_ok=True)

# 2. Build the raw CSV data string with 12 realistic column names
csv_content = """InvoiceID,PurchaseDate,ClientCompany,TotalAmount,PaymentMethod,TaxAmount,ShippingCost,DiscountApplied,StoreLocation,SalesAgent,ProductCategory,Quantity
INV1001,2023-04-12,Alpha Ltd,1250.00,Credit Card,100.00,25.00,0.00,New York,John Doe,Electronics,5
INV1002,2024-08-23,Beta LLC,890.50,PayPal,71.24,15.00,10.00,Chicago,Jane Smith,Office Supplies,12
INV1003,2025-11-02,Gamma Corp,3400.00,Bank Transfer,272.00,0.00,50.00,Houston,Bob Johnson,Furniture,2
INV1004,2026-01-15,Delta Inc,150.25,Credit Card,12.02,5.99,0.00,Los Angeles,Alice Brown,Software,1
"""

# 3. Write the file natively using Python
with open(file_path, "w", encoding="utf-8") as f:
    f.write(csv_content)

print(f"Sample data file successfully created at: {file_path}")

StatementMeta(, ef97a23e-cae7-4556-9b76-dc335261e5c5, 5, Finished, Available, Finished, False)

Sample data file successfully created at: /lakehouse/default/Files/Transactions_raw.csv


In [4]:
from pyspark.sql.functions import year

df = (spark.read
      .option("header", "true")
      .csv("Files/Transactions_raw.csv")
      .select("InvoiceID", "PurchaseDate", "ClientCompany", "TotalAmount")
      .withColumn("PurchaseDate", year("PurchaseDate"))
     )

df.write.format("delta").mode("overwrite").save("Tables/Transactions_cleansed")

StatementMeta(, ef97a23e-cae7-4556-9b76-dc335261e5c5, 6, Finished, Available, Finished, False)

In [5]:
# 1. Generate the execution plan
df.explain(True)

StatementMeta(, ef97a23e-cae7-4556-9b76-dc335261e5c5, 7, Finished, Available, Finished, False)

== Parsed Logical Plan ==
'Project [InvoiceID#743, year('PurchaseDate) AS PurchaseDate#772, ClientCompany#745, TotalAmount#746]
+- Project [InvoiceID#743, PurchaseDate#744, ClientCompany#745, TotalAmount#746]
   +- Relation [InvoiceID#743,PurchaseDate#744,ClientCompany#745,TotalAmount#746,PaymentMethod#747,TaxAmount#748,ShippingCost#749,DiscountApplied#750,StoreLocation#751,SalesAgent#752,ProductCategory#753,Quantity#754] csv

== Analyzed Logical Plan ==
InvoiceID: string, PurchaseDate: int, ClientCompany: string, TotalAmount: string
Project [InvoiceID#743, year(cast(PurchaseDate#744 as date)) AS PurchaseDate#772, ClientCompany#745, TotalAmount#746]
+- Project [InvoiceID#743, PurchaseDate#744, ClientCompany#745, TotalAmount#746]
   +- Relation [InvoiceID#743,PurchaseDate#744,ClientCompany#745,TotalAmount#746,PaymentMethod#747,TaxAmount#748,ShippingCost#749,DiscountApplied#750,StoreLocation#751,SalesAgent#752,ProductCategory#753,Quantity#754] csv

== Optimized Logical Plan ==
Project [I